# Ersatzprüfung - Arbeitspaket (AP) 2: Management & Nutzung Temporale Daten

### Persönliche Angaben (bitte ergänzen)

<table>
  <tr>
    <td>Vorname:</td>
    <td></td>
  </tr>
  <tr>
    <td>Nachname:</td>
    <td></td>
  </tr>
  <tr>
    <td>Immatrikulationsnummer:</td>
    <td></td>
  </tr>
  <tr>
    <td>Modul:</td>
    <td>Data Science</td>
  </tr>
  <tr>
    <td>Prüfungsdatum / Raum / Zeit:</td>
    <td>14.01.2026 / Raum: MU O2.001 / Beginn: 8:00 Uhr</td>
  </tr>
  <tr>
    <td>Erlaubte Hilfsmittel:</td>
    <td>w.MA.XX.DS.25HS (Data Science)<br>Open Book, Eigener Computer, Internet-Zugang</td>
  </tr>
  <tr>
  <td>Nicht erlaubt:</td>
  <td>Nicht erlaubt ist der systematische Einsatz beliebiger Formen von generativer KI (z.B. Copilot, ChatGPT, Google’s «Übersicht mit KI») für das Lösen der Aufgaben sowie beliebige Formen von Kommunikation oder Kollaboration mit anderen Menschen.</td>
</tr>
</table>

## Bewertungskriterien

### <b style="color: gray;">(max. erreichbare Punkte: 32)</b>

<table>
  <thead>
    <tr>
      <th>Kategorie</th>
      <th>Beschreibung</th>
      <th>Punkteverteilung</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Code nicht lauffähig oder Ergebnisse nicht sinnvoll</td>
      <td>Der Code enthält Fehler, die verhindern, dass er ausgeführt werden kann (z.B. Syntaxfehler) oder es werden Ergebnisse ausgegeben, welche nicht zur Fragestellung passen.</td>
      <td>0 Punkte</td>
    </tr>
    <tr>
      <td>Code lauffähig, aber mit gravierenden Mängeln</td>
      <td>Der Code läuft, aber die Ergebnisse sind aufgrund wesentlicher Fehler unvollständig (z.B. fehlende Joins, gravierende Fehler in SQL-Abfragen). Nur geringer Fortschritt erkennbar.</td>
      <td>25% der max. erreichbaren Punkte</td>
    </tr>
    <tr>
      <td>Code lauffähig, aber mit mittleren Mängeln</td>
      <td>Der Code läuft und liefert teilweise korrekte Ergebnisse, aber es gibt grössere Fehler (z.B. fehlende Spalten, unvollständige SQL-Abfragen). Die Ergebnisse sind nachvollziehbar, aber unvollständig oder ungenau.</td>
      <td>50% der max. erreichbaren Punkte</td>
    </tr>
    <tr>
      <td>Code lauffähig, aber mit minimalen Mängeln</td>
      <td>Der Code läuft und liefert ein weitgehend korrektes Ergebnis, aber kleinere Fehler (z.B. falsche oder fehlende Sortierung, Rundung von Werten falsch) beeinträchtigen die Vollständigkeit des Ergebnisses.</td>
      <td>75% der max. erreichbaren Punkte</td>
    </tr>
    <tr>
      <td>Code lauffähig und korrekt</td>
      <td>Der Code läuft einwandfrei und liefert das korrekte Ergebnis ohne Mängel.</td>
      <td>100% der max. erreichbaren Punkte</td>
    </tr>
  </tbody>
</table>



## <b>Vorbereitung (Dieser Teil wird <u>nicht</u> bewertet!)</b>

#### <b>Beschreibung des Datensatzes `synthetic_food_delivery_orders.csv`</b>

- **Inhalt:** Synthetische E-Commerce-Bestellereignisse (Januar 2024 – Dezember 2025)
- **Schema:**
  - `time` (lokale Zeit, Europe/Zurich, Format `YYYY-MM-DD HH:MM:SS`)
  - `courier_id` (int)
  - `restaurant_id` (int)
  - `city` (category: Zürich, Bern, Basel, Lausanne, Genf, Luzern, St. Gallen)
  - `status` (category: placed, picked_up, delivered, cancelled, failed_delivery)
  - `order_value` (float, CHF) – Warenkorbwert
  - `delivery_fee` (float, CHF) – Liefergebühr
  - `tip (float)` – Trinkgeld (in der Regel 0 bei stornierten Bestellungen)
  - `distance_km (float)` – Distanz Restaurant → Kunde
  - `prep_time_min (int)` – Zubereitungszeit
  - `delivery_time_min (int)` – Lieferzeit (bei cancelled kann 0 vorkommen)
- **Hinweise zur Nutzung:**
  - Für **Abfragen** mit Tinyflux Zeitstempel nach **UTC** konvertieren
- **Ziel:**
  - Beschreibung des Datensatzes
  - Korrekte Zuordnung von Time, Tags und Fields
  - Einlesen, in Tinyflux schreiben
  - Zeitfenster-Abfragen und Aggregationen

#### <b>1.) Python Libraries und Settings importieren:</b>

In [1]:
# Libraries
import os
import pandas as pd
import csv
from tinyflux import TinyFlux, Point, FieldQuery, TagQuery, TimeQuery
from datetime import datetime, timezone, timedelta

# Settings
import warnings
warnings.filterwarnings("ignore")

# Abkürzungen für Query-Typen
time = TimeQuery()
tags = TagQuery()
field = FieldQuery()

#### <b>2.) Laden Sie die Funktion für die Erstellung eines Dataframes aus Query-Resultaten in Ihren Arbeitsspeicher:</b>

In [2]:
def points_to_dataframe(points_list, prefer_union=True):
    """
    Wandelt TinyFlux-Query-Resultate (Liste von Point) in ein DataFrame um, ohne dass
    Tags/Fields manuell gemappt werden müssen.

    Verhalten:
    - Wenn points_list nicht leer ist: Schema wird aus points_list abgeleitet und gecacht.
    - Wenn points_list leer ist: es wird zuerst ein gecachtes Schema verwendet.
      Falls kein Cache vorhanden ist, wird versucht, aus einer globalen TinyFlux-DB-Instanz `db`
      einen Sample-Point zu lesen, um das Schema zu inferieren.
    - Wenn alles fehlschlägt: leeres DF nur mit 'time' (weil Schema nicht ableitbar ist).
    """

    # --- Helper: Schema aus Points ableiten ---
    def _infer_schema(points):
        if not points:
            return [], []
        if prefer_union:
            tag_keys = sorted({k for p in points for k in getattr(p, "tags", {}).keys()})
            field_keys = sorted({k for p in points for k in getattr(p, "fields", {}).keys()})
        else:
            tag_keys = sorted(getattr(points[0], "tags", {}).keys())
            field_keys = sorted(getattr(points[0], "fields", {}).keys())
        return tag_keys, field_keys

    # --- Cache initialisieren (persistiert über Funktionsaufrufe) ---
    if not hasattr(points_to_dataframe, "_schema_cache"):
        points_to_dataframe._schema_cache = {"tag_keys": None, "field_keys": None}

    cache = points_to_dataframe._schema_cache

    # --- Fall 1: Query liefert Punkte -> Schema direkt inferieren und cachen ---
    if points_list:
        tag_keys, field_keys = _infer_schema(points_list)
        cache["tag_keys"], cache["field_keys"] = tag_keys, field_keys

    # --- Fall 2: Query ist leer -> Cache nutzen oder (optional) DB sampeln ---
    else:
        tag_keys = cache.get("tag_keys")
        field_keys = cache.get("field_keys")

        if tag_keys is None or field_keys is None:
            # Versuch: globales `db` verwenden (falls vorhanden) und 1+ Punkte lesen
            sample_points = None
            _db = globals().get("db", None)

            if _db is not None:
                # Wir versuchen mehrere gängige Möglichkeiten, ohne die Import-/DB-Logik zu ändern.
                # (Je nach tinyflux-Version gibt es unterschiedliche Methoden.)
                for getter in ("all", "read_all", "select", "__iter__"):
                    try:
                        if getter == "__iter__" and hasattr(_db, "__iter__"):
                            sample_points = list(_db)
                        elif hasattr(_db, getter):
                            candidate = getattr(_db, getter)()
                            sample_points = list(candidate) if candidate is not None else None
                        if sample_points:
                            break
                    except Exception:
                        continue

            if sample_points:
                tag_keys, field_keys = _infer_schema(sample_points)
                cache["tag_keys"], cache["field_keys"] = tag_keys, field_keys
            else:
                print(
                    "Die Query hat keine Daten geliefert. "
                    "Ein Schema konnte nicht inferiert werden (kein Cache und kein DB-Sample verfügbar)."
                )
                return pd.DataFrame(columns=["time"])

        # Wenn wir hier sind, haben wir (mindestens) Schema aus Cache oder DB-Sample
        columns = ["time"] + list(tag_keys) + list(field_keys)
        print("Die Query hat keine Daten geliefert (leeres Resultat).")
        return pd.DataFrame(columns=columns)

    # --- DataFrame aus points_list aufbauen ---
    tag_keys = cache["tag_keys"] or []
    field_keys = cache["field_keys"] or []
    columns = ["time"] + list(tag_keys) + list(field_keys)

    rows = []
    for p in points_list:
        row = {"time": getattr(p, "time", None)}
        tags = getattr(p, "tags", {}) or {}
        fields = getattr(p, "fields", {}) or {}

        row.update({k: tags.get(k) for k in tag_keys})
        row.update({k: fields.get(k) for k in field_keys})
        rows.append(row)

    df = pd.DataFrame(rows).reindex(columns=columns)
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    return df

#### <b>3.) Starten Sie eine GitHub Codespaces Instanz auf Basis Ihres Forks des folgenden GitHub Repositories:</b>

##### GitHub-Repository: https://github.com/MariaPelli/tinyflux


#### <b>4.) Erstellen Sie mit Tinyflux eine Datenbank mit dem Namen "delivery.db".</b>

In [3]:
# Erstellung der Tinyflux-Datenbank
db = TinyFlux("delivery.db")

## <b>Aufgaben: Einlesen der Daten in die Datenbank (Dieser Teil wird bewertet!)</b>

#### <b>Aufgabe (1): Lesen Sie die CSV-Datei 'synthetic_food_delivery_orders.csv' in ein Dataframe mit dem Namen "df" ein, zeigen Sie die ersten 10 Records und geben deskriptive Statistiken des Dataframes aus.</b>

<b>Details zur Aufgabenstellung:</b>

- Das Laden der CSV-Datei kann mittels der Pandas-Funktion "read_csv" (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) erfolgen
- Achten Sie darauf, den Zeitstempel mittels der Pandas-Funktion __df['timestamp']= pd.to_datetime(df['timestamp'], format='mixed')__ auf das Format datetime zu setzen (https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- Die ersten 10 Records können mittels der Pandas-Funktion "df.head()" wiedergegeben werden (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html) 
- Deskriptive Statistiken können mittels Pandas-Funktion "df.describe()" (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) wiedergegeben werden. Geben Sie diese für quantitative und kategoriale Variablen aus, indem Sie den Parameter include auf 'all' setzen.

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [4]:
# CSV-Datei in Dataframe laden


# Das Format des Zeitstempels in Pandas setzen



In [5]:
# Erste 10 Records des Dataframes anzeigen


In [6]:
# Deskriptive Statistiken des Dataframes


#### <b>Aufgabe (2): Schreiben Sie den Inhalt der CSV-Datei 'synthetic_food_delivery_orders.csv' in die erstellte Tinyflux-Datenbank "delivery.db" und weisen sie Time, Fields und Tags zu.</b>

<b style="color: gray;">(max. erreichbare Punkte: 10)</b>

In [7]:
# Tinyflux Datenbank leeren
os.path.exists('delivery.db') and open('delivery.db','w').close()

# Einlesen der CSV-Datei in Tinyflux


<div>

#### <b> Hinweis! 

#### Falls etwas mit der Erstellung der Datenbank nicht funktioniert, verwenden Sie für die folgenden Aufgaben die Tinyflux Datenbank 'delivery_fallback.db'. Gehen Sie davon aus, dass diese unter denselben, obigen Bedingungen erstellt wurde.</b>
    
#### <b>Erstellen der Verbindung mit der Tinyflux-Datenbank mit dem Namen "delivery_fallback.db":</b>
</div>

In [8]:
# Erstellung der Tinyflux-Datenbank (wird bereitgestellt)
db = TinyFlux("delivery_fallback.db")

## <b>Aufgaben: Abfragen (Queries) (Dieser Teil wird bewertet!)</b>

<b >Hinweise zu den folgenden Aufgabenstellungen:</b>
<ul>
  <li>Erstellen Sie je Aufgabe eine Time-, Field- oder Tag-Abfrage bzw. eine Kombination von diesen und integrieren Sie diese in Ihren Python Code.</li>
  <li>Speichern Sie je Aufgabe die Ergebnistabelle mittels der zu Beginn dieses Notebooks bereitgestellten Funktion <b>points_to_dataframe</b> in einem DataFrame und stellen Sie diesen bzw. dessen Eigenschaften ganz oder in Teilen (z.B. erste 5 Zeilen) dar.</li>
</ul>

#### <b>Aufgabe (3): Geben Sie alle Bestellungen aus **`Zuerich`** im Zeitraum **01.06.2024 00:00 UTC bis 08.06.2024 00:00 UTC** aus, die den Status **`delivered`** haben.</b>
    
<b>Details zur Aufgabenstellung:</b>
- DataFrame erstellen und die ersten 10 Records anzeigen
    
<b style="color: gray;">(max. erreichbare Punkte: 6)</b>

In [9]:
# Definieren der Query


# Datenabfrage mittels der definierten Query


# Erstellung eines Dataframes aus den Query-Resultaten und Ausgabe der ersten 10 Records



#### <b>Aufgabe (4): Ermitteln Sie das durchschnittliche Trinkgeld (**`tip`**) pro Jahr für mind. 2 Städte Ihrer Wahl (**`city`**) über alle Bestellungen, ausgenommen der Stornierten (**`cancelled`**)</b>
    
<b>Details zur Aufgabenstellung:</b>
- Alle in einem kategorialen Attribut vorhandenen Kategorien können Sie mittels der Pandas-Funktion "df.attributname.unique()" (https://pandas.pydata.org/docs/reference/api/pandas.unique.html) ermitteln
- Den Durchschnitt über eine Spalte können Sie mittels der Pandas-Funktion df[['spaltenname']].mean() https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html ermitteln
- "!=" ist in Python der Operator für „ungleich“
    
<b style="color: gray;">(max. erreichbare Punkte: 12)</b>

In [10]:
# Ermittlung aller vorhandenen Städte


In [11]:
# Definieren der Queries


# Erstellung eines Dataframes aus den Query-Resultaten


# Ausgabe der aggregierten Resultate



---
### Notebook Footer
*Bitte belassen Sie diesen Abschnitt im Notebook.*

- Python Version
- Paket-Versionen (tinyflux, pandas)
- Datum/Uhrzeit der Ausführung

In [12]:
import tinyflux
import sys, pandas as pd
from importlib.metadata import version

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("tinyflux:", version("tinyflux"))
print("UTC now:", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S %Z"))


Python: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
pandas: 2.2.3
tinyflux: 1.0.0
UTC now: 2026-01-07 07:06:07 UTC
